# Day 10 : Feature Engineering and Cross Validation

In [46]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

df = pd.read_csv('train.csv')

## Question 1 - Baseline Model

In [38]:
X = df[['Pclass','Age','Fare','SibSp','Parch']].copy()
y = df['Survived'].copy()

X['Median_Age'] = X.groupby('Pclass')['Age'].transform('median')
X['Age'].fillna(X['Median_Age'], inplace=True)
#print(X.isna().sum())
X.drop(columns=['Median_Age'], inplace=True)

X_train,X_test,y_train,y_test = train_test_split(X,y, test_size=0.2, random_state=42)
# print(X_train.shape)
# print(X_test.shape)

model = RandomForestClassifier(n_estimators=100,random_state=42)
model.fit(X_train,y_train)

base_pred = model.predict(X_test)

base_acc = accuracy_score(np.array(y_test), base_pred)
base_precision = precision_score(np.array(y_test), base_pred)
base_recall = recall_score(np.array(y_test), base_pred)
base_f1 = f1_score(np.array(y_test), base_pred)

print(f"Base:\n1. Accuracy = {base_acc}\n2. Precision = {base_precision}\n3. Recall = {base_recall}\n4. F1 = {base_f1}")

Base:
1. Accuracy = 0.7597765363128491
2. Precision = 0.7384615384615385
3. Recall = 0.6486486486486487
4. F1 = 0.6906474820143885


## Question 2 - Feature Engineering

In [ ]:
X['FamilySize'] = X['SibSp'] + X['Parch'] +1
X['IsAlone'] = [1 if X['FamilySize'][i] == 1 else 0 for i in range(len(X))]  # needed help with inline if syntax lol
## Correction
# X["IsAlone"]=(X['FamilySize']==1).astype(int)
X.head()

,Pclass,Age,Fare,SibSp,Parch,FamilySize,IsAlone
0,3,22.0,7.2500,1,0,2,0
1,1,38.0,71.2833,1,0,2,0
2,3,26.0,7.9250,0,0,1,1
3,1,35.0,53.1000,1,0,2,0
4,3,35.0,8.0500,0,0,1,1


1. They may be useful because they may provide more information to the model to predict survival.
2. Because it provides a more comprehensive feature rather than many related features i guess.

## Question 3 - Improved Model

In [40]:
X_new = X.drop(columns=['SibSp','Parch']).copy()

X_train,X_test,y_train,y_test = train_test_split(X_new,y, test_size=0.2, random_state=42)
# print(X_train.shape)
# print(X_test.shape)

model = RandomForestClassifier(n_estimators=100,random_state=42)
model.fit(X_train,y_train)

imp_pred = model.predict(X_test)

imp_acc = accuracy_score(np.array(y_test), imp_pred)
imp_precision = precision_score(np.array(y_test), imp_pred)
imp_recall = recall_score(np.array(y_test), imp_pred)
imp_f1 = f1_score(np.array(y_test), imp_pred)

print(f"Improved:\n1. Accuracy = {imp_acc}\n2. Precision = {imp_precision}\n3. Recall = {imp_recall}\n4. F1 = {imp_f1}")

Improved:
1. Accuracy = 0.7597765363128491
2. Precision = 0.7313432835820896
3. Recall = 0.6621621621621622
4. F1 = 0.6950354609929078


It doesnt seem to have helped with accuracy, probably because the features were derived from 2 similar correlated features which were already part of the dataset.

Recall and F1 did increase which is a good thing but the accuracy stayed the same.

**Apparently important part wasnt accuracy but that i compared raw features with engineered features**

## Question 4 - Feature Importance

In [42]:
feat_imp = model.feature_importances_
feat_df = pd.DataFrame({
    'Feature': X_new.columns,
    'Importance': feat_imp
})
feat_df.sort_values(by='Importance',ascending=False,inplace=True)
feat_df

,Feature,Importance
2,Fare,0.434609
1,Age,0.394639
0,Pclass,0.088570
3,FamilySize,0.065841
4,IsAlone,0.016341


Not really, the engineered features are the least important as compared to the rest.

## Question 5 - Correlation Check

In [45]:
feat_corr = X_new.corr()
feat_corr

,Pclass,Age,Fare,FamilySize,IsAlone
Pclass,1.000000,-0.408487,-0.549500,0.065997,0.135207
Age,-0.408487,1.000000,0.123784,-0.251918,0.165356
Fare,-0.549500,0.123784,1.000000,0.217138,-0.271832
FamilySize,0.065997,-0.251918,0.217138,1.000000,-0.690922
IsAlone,0.135207,0.165356,-0.271832,-0.690922,1.000000


None of the features have high positive correlation. Although FamilSize and isAlone have high negative correlation at ```-0.69``` and Pclass and Fare have high negative correlation at ```-0.55```

This signifies strong inverse relationships among these features, which is kind of a given if you know what these features are. Since there is no high positive correlations, it does not fall into possibility of multicolinearity.

## Question 6 - K-Fold Cross Validation

In [49]:
scores = cross_val_score(model,X_new,y,cv=5)
print(f"Cross Val Scores = {scores}")
print(f"Cross Val Mean Scores = {scores.mean()}")

Cross Val Scores = [0.67039106 0.66292135 0.67977528 0.73595506 0.70224719]
Cross Val Mean Scores = 0.6902579875714017


It is better because it provides more reliable and unbiased estimates of how a model will perform on unseen data. It also detects overfitting and is better for hyperparameter tuning.

## Question 7 - Model Stability

In [50]:
print(f"Cross Val Std = {scores.std()}")

Cross Val Std = 0.026396351327302177


Large standard deviation suggests that the possibility of noisy data as well as outliers in the data which could introduce biases and therefore skew the predictions.

**Correction : Large Std means model performance changes significantly depending on the split, suggesting instability**

## Question 8 - Data Leakage

Survived_Last_Year would provide a sort of bias to the model, since x survived last year the odds of them surviving now also increases, but that might not always be the case. Since the situations are different for their survival in both scenarios. But the model doesnt know that and will skew the predictions in favor of those that survived in the previous year.

**I was right about look ahead bias lol**

## Question 9 - Feature Selection

In [53]:
few_df = X_new[['Fare','FamilySize','IsAlone']].copy()

X_train,X_test,y_train,y_test = train_test_split(few_df,y, test_size=0.2, random_state=42)
# print(X_train.shape)
# print(X_test.shape)

model = RandomForestClassifier(n_estimators=100,random_state=42)
model.fit(X_train,y_train)

few_pred = model.predict(X_test)

few_acc = accuracy_score(np.array(y_test), few_pred)
few_precision = precision_score(np.array(y_test), few_pred)
few_recall = recall_score(np.array(y_test), few_pred)
few_f1 = f1_score(np.array(y_test), few_pred)

print(f"Improved:\n1. Accuracy = {few_acc}\n2. Precision = {few_precision}\n3. Recall = {few_recall}\n4. F1 = {few_f1}")

Improved:
1. Accuracy = 0.6871508379888268
2. Precision = 0.65
3. Recall = 0.527027027027027
4. F1 = 0.582089552238806


Fewer features can perform just as well in some cases when those features are great features that have a decent correlation to the target and do not have any outliers or noise. Depending on the dataset and model, it is very possible. Few great features are better than alot of mediocre features, it just introduces noise.

## Question 10 - Quant Thinking

Firstly, apart from the given features, i could possibly use in built libraries like TA-Lib to generate features and then conduct feature reduction or selection.
But from the features I have, I could possibly derive multiple features such as implied vol, vwap, orderbook liquidity, etc.

In Quant Research, you must find patterns and features in data which cannot be found by a lay man, we need to find those prime variables which could give us an edge in the fast paced markets.

## Bonus ML Question

1. Yes the model is actually very good
2. The model is not overfitting or underfitting, it seems to have understood the patterns in the data thoroughly and even the cross validation is providing us with the confirmation.
3. Yes, because in the latter, the model is definitely overfitting on the training data which is why it is unable to provide a better test accuracy.

---

## DSA Challenge #5

In [55]:
# I googled this cuz i went blank

arr = [2,1,5,1,3,2]
k = 3
max_len = len(arr)

window_sum = sum(arr[:k])
max_sum = window_sum

for i in range(max_len-k):
    window_sum = window_sum-arr[i]+arr[i+k]
    max_sum = max(window_sum,max_sum)

print (max_sum)
    

9


1. O(n)^2  **Correction O(n*k)**
2. O(n)
3. Reduces time complexity and avoids nested loops

## DSA Challenge # 6

In [ ]:
nums = [1,2,3,4,6]
target=6
left,right=0,0
max_len = len(nums)
num_sum = 0

for left in range(max_len):
    right = left+1
    while right < max_len:
        num_sum = nums[left]+nums[right]
        if num_sum==target:
            print([nums[left],nums[right]])
        right+=1
    left+=1

## Correction

left = 0
right = len(nums)-1

while left<right:
    curr_sum = nums[left]+nums[right]
    if curr_sum == target:
        print([nums[left],nums[right]])
        break
    elif curr_sum < target:
        left+=1
    else:
        right-=1

## Basically the first list should probably be sorted


[2, 4]
[2, 4]


1. Because we are traversing through an array 
2. I donno dawg
3. supposedly O(n)